# **05 최종 실습**

### 필수 요구사항

1. **필수 미들웨어 2개 적용**
   - TodoListMiddleware (작업 추적)
   - SummarizationMiddleware (대화 요약)

2. **커스텀 미들웨어 2개 정의**
   - 에이전트 동작 제어
   - 로깅, 모니터링, 검증 등

3. **프롬프트 엔지니어링**
   - 에이전트 목적 정의
   - System prompt 작성
   - 페르소나 설정

## 1. 환경 설정

- OpenAI API Key 발급: https://platform.openai.com/api-keys

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI API Key가 설정되었습니다.")

## 2. 프로젝트 개요

### 예시 프로젝트 아이디어

다음은 구현가능한 에이전트 예시입니다. 참고하거나 본인만의 아이디어를 구현하세요.

1. **📚 연구 보조 에이전트**
   - 논문 검색, 요약, 참고문헌 관리
   - 도구: 웹 검색, 계산기, 파일 읽기/쓰기

2. **📊 데이터 분석 에이전트**
   - 데이터 수집, 분석, 시각화
   - 도구: 계산기, 통계 함수, 그래프 생성

3. **🛒 쇼핑 비서 에이전트**
   - 제품 검색, 가격 비교, 추천
   - 도구: 웹 검색, 가격 계산, 리뷰 분석

4. **📅 일정 관리 에이전트**
   - 일정 추가, 알림, 우선순위 관리
   - 도구: 날짜 계산, 일정 CRUD, 알림

5. **💬 고객 상담 에이전트**
   - FAQ 응답, 문제 해결, 티켓 생성
   - 도구: 검색, 티켓 시스템, 이메일

In [ ]:
# 프로젝트 정의
PROJECT_NAME = "연구 보조 에이전트"  # 여기를 수정하세요
PROJECT_DESCRIPTION = "논문을 검색하고 요약하며, 연구 노트를 관리하는 에이전트"  # 여기를 수정하세요

print(f"에이전트: {PROJECT_NAME}")
print(f"설명: {PROJECT_DESCRIPTION}")

## 3. 기본 도구 정의

에이전트가 사용할 도구들을 정의합니다. 아래는 예시이며, 필요에 따라 수정하세요.

In [ ]:
from langchain.tools import tool
from datetime import datetime

@tool
def search_web(query: str) -> str:
    """Search the web for information. Returns relevant results."""
    # 실제로는 Tavily API 등을 사용
    return f"Search results for '{query}': [모의 검색 결과]\n1. 관련 정보 1\n2. 관련 정보 2\n3. 관련 정보 3"

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Use for calculations."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def save_note(title: str, content: str) -> str:
    """Save a note with a title and content."""
    # 실제로는 파일이나 데이터베이스에 저장
    return f"Note '{title}' saved successfully with {len(content)} characters."

# 도구 리스트
tools = [search_web, calculator, get_current_time, save_note]

for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

## 4. 필수 미들웨어 적용

TodoListMiddleware와 SummarizationMiddleware를 적용합니다.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware, SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# 필수 미들웨어 설정
required_middleware = [
    # TodoList: 작업 추적 및 계획
    TodoListMiddleware(),

    # Summarization: 긴 대화 요약
    SummarizationMiddleware(
        model="gpt-4o-mini",
        trigger=("tokens", 3000),  # 3000 토큰 도달 시 요약
        keep=("messages", 15),      # 최근 15개 메시지 유지
    ),
]

print("  - TodoListMiddleware: 작업 추적")
print("  - SummarizationMiddleware: 대화 요약")

## 5. 기본 에이전트 생성 및 테스트

필수 미들웨어만 적용한 기본 에이전트를 만들고 테스트합니다.

In [ ]:
# 기본 에이전트 생성
basic_agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
    checkpointer=InMemorySaver(),
    middleware=required_middleware,
    system_prompt="You are a helpful AI assistant.",
)

In [ ]:
# 기본 에이전트 테스트
config = {"configurable": {"thread_id": "basic-test-1"}}

response = basic_agent.invoke(
    {"messages": [{"role": "user", "content": "What is 25 * 4? Also, what time is it now?"}]},
    config=config
)

print("\n=== 기본 에이전트 응답 ===")
print(response['messages'][-1].content)

## 6. 📖 실습: 커스텀 미들웨어 1 정의

### 미션

첫 번째 커스텀 미들웨어를 정의하세요.

### 추천 아이디어

1. **성능 모니터링 미들웨어**
   - 각 단계의 실행 시간 측정
   - 토큰 사용량 추적
   - 비용 계산

2. **입력 검증 미들웨어**
   - 사용자 입력 길이 제한
   - 금지어 필터링
   - 질문 형식 검증

3. **로깅 미들웨어**
   - 모든 모델 호출 기록
   - 도구 사용 통계
   - 에러 로깅

4. **응답 품질 검증 미들웨어**
   - 응답 길이 체크
   - 특정 키워드 포함 확인
   - 응답 형식 검증

### 구현 가이드

아래 템플릿을 사용하여 구현하세요:

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime
from typing import Any

class CustomMiddleware1(AgentMiddleware):
    """첫 번째 커스텀 미들웨어"""

    def __init__(self):
        # CODE HERE: 필요한 속성 초기화
        pass

    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """에이전트 시작 전 실행"""
        # CODE HERE: 에이전트 시작 시 로직 구현
        return None

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """모델 호출 전 실행"""
        # CODE HERE: 모델 호출 전 로직 구현
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """모델 호출 후 실행"""
        # CODE HERE: 모델 호출 후 로직 구현
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """에이전트 종료 시 실행"""
        # CODE HERE: 에이전트 종료 시 로직 구현
        return None

# 미들웨어 인스턴스 생성
# custom_middleware_1 = CustomMiddleware1()

## 7. 📖 실습: 커스텀 미들웨어 2 정의

### 미션

두 번째 커스텀 미들웨어를 정의하세요. 첫 번째와 다른 기능을 구현하세요.

### 데코레이터 기반 미들웨어 옵션

클래스 대신 데코레이터를 사용할 수도 있습니다:

In [ ]:
from langchain.agents.middleware import before_model, after_model

# 옵션 1: 데코레이터 기반
@before_model
def custom_middleware_2a(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """모델 호출 전 실행"""
    # CODE HERE: 로직 구현
    return None

@after_model
def custom_middleware_2b(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """모델 호출 후 실행"""
    # CODE HERE: 로직 구현
    return None

# 또는 옵션 2: 클래스 기반 (미들웨어 1과 동일한 패턴)
class CustomMiddleware2(AgentMiddleware):
    """두 번째 커스텀 미들웨어"""

    def __init__(self):
        # CODE HERE
        pass

    # CODE HERE: 필요한 훅 메서드 구현

# 미들웨어 인스턴스 생성
# custom_middleware_2 = CustomMiddleware2()
# 또는
# custom_middleware_2 = [custom_middleware_2a, custom_middleware_2b]

## 8. 📖 실습: 프롬프트 엔지니어링

### 미션

에이전트의 System Prompt를 작성하세요.

### 프롬프트 작성 가이드

좋은 System Prompt는 다음을 포함합니다:

1. **역할 정의**: "당신은..."
2. **목표**: "목표는 ..."
3. **행동 지침**: "~와 같은 상황에서는, ..."
4. **제약사항**: "절대 ...", "항상 ..."
5. **톤/스타일**: "Be professional/friendly/concise"
6. **XML 태그(<></>)**: 구조화 작성

In [ ]:
# System Prompt 작성
SYSTEM_PROMPT = """
# CODE HERE: 자신만의 System Prompt를 작성하세요

예시:

당신은 학술 논문과 과학 문헌에 특화된 전문 연구 보조 에이전트입니다.

<goals>
- 논문 검색 및 이해 지원
- 복잡한 학술 내용을 명확한 언어로 요약
- 연구 노트와 참고 문헌 정리
</goals>

<guidelines>
- 항상 출처를 명시하세요
- 최신 정보를 찾기 위해 search_web 도구를 사용하세요
- 중요한 발견을 save_note 도구를 사용하여 저장하세요
- 복잡한 작업을 todo list를 사용하여 분해하세요
</guidelines>

<constraints>
- 출처를 조작하거나 가짜 결과를 만들지 마세요
- 확실하지 않은 경우, 명확히 하기 위해 질문하세요
- 요약을 정확하고 간결하게 유지하세요
</constraints>

<Tone>
적절하게 전문적이면서도 접근하기 쉽고, 명확하며 교육적인 톤을 유지하세요.
</Tone>
""".strip()

## 9. 📖 실습: 최종 에이전트 구성

### 미션

모든 구성 요소를 결합하여 최종 에이전트를 만드세요.

- 필수 미들웨어
- 커스텀 미들웨어 1
- 커스텀 미들웨어 2
- System Prompt

In [ ]:
# 모든 미들웨어 결합
all_middleware = [
    # CODE HERE: 필수 미들웨어 추가
    # required_middleware[0],  # TodoList
    # required_middleware[1],  # Summarization

    # CODE HERE: 커스텀 미들웨어 추가
    # custom_middleware_1,
    # custom_middleware_2,
]

# 최종 에이전트 생성
# CODE HERE: create_agent 함수 호출
# final_agent = create_agent(
#     model="gpt-4o-mini",
#     tools=tools,
#     checkpointer=InMemorySaver(),
#     middleware=all_middleware,
#     system_prompt=SYSTEM_PROMPT,
# )

# print(f"  - 도구: {len(tools)}개")
# print(f"  - 미들웨어: {len(all_middleware)}개")
# print(f"  - System prompt: {len(SYSTEM_PROMPT)} 문자")

## 10. 최종 에이전트 테스트

완성된 에이전트를 테스트합니다.

In [ ]:
# 테스트 시나리오 1: 간단한 질문
config = {"configurable": {"thread_id": "final-test-1"}}

# CODE HERE: 에이전트 실행
# response = final_agent.invoke(
#     {"messages": [{"role": "user", "content": "여기에 테스트 질문 입력"}]},
#     config=config
# )

# print("\n=== 테스트 1 응답 ===")
# print(response['messages'][-1].content)

In [ ]:
# 테스트 시나리오 2: 복잡한 작업
# CODE HERE: 여러 단계가 필요한 복잡한 작업 테스트

# 예시:
# response = final_agent.invoke(
#     {
#         "messages": [{
#             "role": "user",
#             "content": "복잡한 다단계 작업 요청"
#         }]
#     },
#     config=config
# )

# print("\n=== 테스트 2 응답 ===")
# print(response['messages'][-1].content)

In [ ]:
# 테스트 시나리오 3: 미들웨어 동작 확인
# CODE HERE: 커스텀 미들웨어가 제대로 작동하는지 확인할 수 있는 테스트

# 예시:
# - 로깅 미들웨어: 로그가 제대로 기록되는가?
# - 검증 미들웨어: 잘못된 입력이 차단되는가?
# - 모니터링 미들웨어: 통계가 정확하게 수집되는가?

## 11. 데모 및 평가

### 데모 스크립트

강사나 동료에게 프로젝트를 시연할 때 사용할 샘플 대화를 작성하세요.

In [ ]:
# 데모 시나리오
demo_config = {"configurable": {"thread_id": "demo-1"}}

demo_queries = [
    # CODE HERE: 3-5개의 데모 시나리오 작성
    # "첫 번째 데모 질문",
    # "두 번째 데모 질문",
    # "세 번째 데모 질문",
]

# for i, query in enumerate(demo_queries, 1):
#     print(f"\n{'='*60}")
#     print(f"데모 {i}: {query}")
#     print('='*60)
#
#     response = final_agent.invoke(
#         {"messages": [{"role": "user", "content": query}]},
#         config=demo_config
#     )
#
#     print("\n응답:")
#     print(response['messages'][-1].content)
#     print()

**※ 본 노트북 파일은 완성 후 모두 실행하여 저(박나연 강사)에게 SLACK DM으로 제출합니다.**